# `RAG_QUERY_JOB` (하이브리드 재무 질의응답 RAG 파이프라인)

- **Recipe File**: `jobs/rag_pipeline.py`
- **Job ID**: `rag_query`
- **Target Queue**: `workflow-core`
- **Version**: `4`

## 1. 개요 및 파이프라인 흐름
복합 자연어 재무 질문을 시점·항목별 서브쿼리로 분해하고, pgvector HNSW 밀집 검색과 PostgreSQL FTS 키워드 검색을 병렬로 수행한 뒤 RRF(Reciprocal Rank Fusion)로 융합하고 컨텍스트를 확장하여 최종 근거 기반 답변을 생성하는 핵심 DAG입니다.

```
  [query] (query_input)
     │
     ▼
  [decompose] (decomposer) ──┐
     │ (subqueries)          │
     ▼                       ▼
  [route] (llm_query_router) ◄── [data-scope] (pgvector_data_scope)
     ├──────────────────────┐
     ▼                      ▼
  [embed-query] (embedder) [keyword] (postgres_native_keyword_retriever)
     │                      │
     ▼                      │
  [dense] (pgvector_retriever)│
     │                      │
     ▼                      ▼
     └────────► [fuse] (rrf_fusion) ◄───────┘
                   │
                   ▼
          [expand-context] (pg_context_expander)
                   │
                   ▼
               [read] (reader)
```

In [ ]:
import sys
from pathlib import Path
import json

# Find repository root by walking up from cwd to find jobs/__init__.py
current = Path.cwd()
PROJECT_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "jobs" / "__init__.py").exists():
        PROJECT_ROOT = parent
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find repository root containing jobs/__init__.py")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jobs.rag_pipeline import RAG_QUERY_JOB

print(f"📌 Loaded Job: {RAG_QUERY_JOB.name} ({RAG_QUERY_JOB.job_id})")
print(f"Description: {RAG_QUERY_JOB.description}")
print(f"\n--- Nodes ({len(RAG_QUERY_JOB.nodes)}) ---")
for node in RAG_QUERY_JOB.nodes:
    print(f"  - Node: {node.node_id:<16} | Module: {node.module_type}")

print(f"\n--- Edges ({len(RAG_QUERY_JOB.edges)}) ---")
for edge in RAG_QUERY_JOB.edges:
    print(f"  - [{edge.source}.{edge.source_output}] -> [{edge.target}.{edge.target_input}]")